# recs_011 — Retrieval comparison (baselines vs two-tower)

**Purpose.** Hold **one offline contract** (same Task A examples, slices, personalization) and compare:
- **Baselines:** bi-encoder / heuristic user-item similarity (see `run_retrieval_eval` + `recs_job_eval_retrieval.py`).
- **Candidates:** trained two-tower retrieval variants (same item tower; query tower differs by feature set).

## Candidate definitions (locked)

- **Candidate A (`two_tower_a_raw_text`)**
  - Query/user tower input: current raw review text only (`q_session`).
- **Candidate B (`two_tower_b_raw_plus_mean_train`)**
  - Query/user tower input: current raw review text + `mean_train_history` (from past train reviews only).
- **Candidate C (`two_tower_c_raw_plus_behavior`)**
  - Query/user tower input: current raw review text + `u_behavior`.
- **Candidate D (`two_tower_d_raw_plus_habit`)**
  - Query/user tower input: current raw review text + `u_habit`, where `u_habit = normalize(0.5 * u_behavior + 0.5 * u_reviews)`.
- **Item tower (shared by A/B/C/D)**
  - Game/item text representation used for full-catalog retrieval.

## Playtime feature decisions (locked)

- **Query scalar features** (used with query/user features):
  - `log1p(query_playtime_hours)`
  - `log1p(author.playtime_last_two_weeks)`
- **History weighting for behavior**:
  - Build `u_behavior` as a playtime-weighted mean of historical game embeddings using train-history only.
  - Per-game weight: `w_i = log1p(hours_i)`.
  - Vector form: `u_behavior = sum(w_i * emb(game_i)) / sum(w_i)`.
- **Scope note**:
  - Apply playtime weighting to `u_behavior` only (not required for `u_reviews`).

This A/B/C/D setup tests incremental value of review-history (`recs_004`), behavior-history, and habit fusion while keeping item-side representation fixed.

**What you implement next (outside this notebook).**
- Training job + artifact writer (e.g. `user_vectors.npz`, `item_vectors.npz`, `meta.json` with alignments to `app_id`).
- A small loader that, for each eval example, returns full-catalog scores compatible with `scores_from_query`-style ranking (or replace with your ANN path but keep the same metric code).

**Notebook role.** Orchestration + QA + tables once artifacts exist. Prefer mirroring output shapes of `eval_retrieval_*.csv` for apples-to-apples diffing.

## Prerequisites

- Run retrieval baseline job:
  - `python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json`
- Artifacts: `artifacts/recs/eval/eval_retrieval_*.csv` and `eval_retrieval_run_meta.json`.
- **Optional:** precomputed candidate overall table `artifacts/recs/eval/eval_retrieval_two_tower_overall.csv` (same columns as baseline `eval_retrieval_overall.csv` for the candidate method).
- **Two-tower artifacts:** (TBD path under `artifacts/recs/` — define when you add training).

In [13]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from steam_review_ml.recommender.evaluation import METRIC_COLS
from steam_review_ml.utils import load_config


def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


REPO_ROOT = _find_repo_root(Path.cwd())

EVAL_JOB_CONFIG = REPO_ROOT / "configs" / "recs_job_eval_retrieval.json"
BASELINE_EVAL_DIR = REPO_ROOT / "artifacts" / "recs" / "eval"

PATH_OVERALL = BASELINE_EVAL_DIR / "eval_retrieval_overall.csv"
PATH_BY_SLICE = BASELINE_EVAL_DIR / "eval_retrieval_by_slice.csv"
PATH_RUN_META = BASELINE_EVAL_DIR / "eval_retrieval_run_meta.json"

# When you precompute a two-tower eval with the same contract, drop a matching CSV here (optional).
PATH_TWO_TOWER_OVERALL = BASELINE_EVAL_DIR / "eval_retrieval_two_tower_overall.csv"

BASELINE_METHODS = ["raw", "popularity_train", "multi_mean_train"]
CANDIDATE_METHOD = "two_tower"

job_cfg = load_config(str(EVAL_JOB_CONFIG))
K_FINAL = int(job_cfg.get("k_final", 10))

In [14]:
for p in (PATH_OVERALL, PATH_BY_SLICE):
    if not p.is_file():
        raise FileNotFoundError(
            f"Missing {p}\n"
            "Run: python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json"
        )

baseline_overall = pd.read_csv(PATH_OVERALL)
baseline_by_slice = pd.read_csv(PATH_BY_SLICE)

run_meta = None
if PATH_RUN_META.is_file():
    run_meta = json.loads(PATH_RUN_META.read_text(encoding="utf-8"))

required = ["method", *METRIC_COLS]
missing = [c for c in required if c not in baseline_overall.columns]
if missing:
    raise ValueError(f"eval_retrieval_overall.csv missing columns: {missing}")

baseline_overall_sub = baseline_overall[baseline_overall["method"].isin(BASELINE_METHODS)].copy()

print("Loaded baseline tables:")
print(" ", PATH_OVERALL, f"rows={len(baseline_overall)}")
print(" ", PATH_BY_SLICE, f"rows={len(baseline_by_slice)}")
if run_meta:
    print(" run_meta keys:", sorted(run_meta.keys())[:12], "...")
print(f" k_final (from job config)={K_FINAL}")

Loaded baseline tables:
  /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval/eval_retrieval_overall.csv rows=3
  /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval/eval_retrieval_by_slice.csv rows=6
 run_meta keys: ['active_cohort', 'config_path', 'counts_by_slice', 'counts_by_support_bucket', 'coverage', 'k_final', 'k_personalization', 'max_examples', 'methods_requested', 'methods_run', 'n_examples_evaluable', 'output_dir'] ...
 k_final (from job config)=10


In [15]:
display(baseline_overall_sub[required].sort_values("NDCG@K", ascending=False))

if PATH_TWO_TOWER_OVERALL.is_file():
    tt = pd.read_csv(PATH_TWO_TOWER_OVERALL)
    miss_tt = [c for c in required if c not in tt.columns]
    if miss_tt:
        raise ValueError(f"Two-tower overall CSV missing columns: {miss_tt}")
    tt_sub = tt[tt["method"] == CANDIDATE_METHOD].copy()
    if tt_sub.empty:
        tt_sub = tt  # accept single-method file
    compare_overall = pd.concat([baseline_overall_sub[required], tt_sub[required]], ignore_index=True)
else:
    placeholder = pd.DataFrame([{"method": CANDIDATE_METHOD, **{m: float("nan") for m in METRIC_COLS}}])
    compare_overall = pd.concat([baseline_overall_sub[required], placeholder], ignore_index=True)

print(
    "\nComparison frame (baseline + candidate). "
    + (
        f"Loaded candidate from {PATH_TWO_TOWER_OVERALL}"
        if PATH_TWO_TOWER_OVERALL.is_file()
        else f"Candidate is NaN placeholder — add {PATH_TWO_TOWER_OVERALL.name} when ready."
    )
)
display(compare_overall)

,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR
0,popularity_train,0.15112,0.146796,0.050594,0.073109,0.073756
1,multi_mean_train,0.08328,0.076479,0.025349,0.037700,0.041694
2,raw,0.07168,0.066606,0.024917,0.035020,0.040381



Comparison frame (baseline + candidate). Candidate is NaN placeholder — add eval_retrieval_two_tower_overall.csv when ready.


,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR
0,popularity_train,0.15112,0.146796,0.050594,0.073109,0.073756
1,multi_mean_train,0.08328,0.076479,0.025349,0.037700,0.041694
2,raw,0.07168,0.066606,0.024917,0.035020,0.040381
3,two_tower,NaN,NaN,NaN,NaN,NaN


### Two-tower scoring (fill in later)

**Contract.** For each eval example, produce a **full-catalog** score vector aligned with `prepare_eval_inputs` / `run_retrieval_eval` (same `app_ids` order as `ContentRetriever`), then reuse the same ranking and metric definitions so `eval_retrieval_*` columns stay comparable.

**Integration options.**

1. **Precomputed table** — Run a small script that writes `eval_retrieval_two_tower_overall.csv` (and optionally by-slice) next to the baseline job outputs; this notebook will auto-pick it up.
2. **In-notebook** — After loading `prepare_eval_inputs(...)` from `steam_review_ml.recommender.evaluation`, define `score_two_tower(ex) -> np.ndarray` and aggregate per-example metrics the same way as `run_retrieval_eval` (today `_per_example_metrics` is module-private; either extend the library with a named helper or keep the runner in a script).

**Artifacts** (TBD): e.g. `artifacts/recs/two_tower/` with vector dumps + `meta.json` mapping ids to `app_id`.

### Define two-tower artifact contract

This cell defines the expected artifact locations and metadata keys for in-notebook evaluation. We keep this explicit so failures are actionable (missing file/key tells you exactly what to export from training).

In [16]:
TWO_TOWER_DIR = REPO_ROOT / "artifacts" / "recs" / "two_tower"

# Optional save paths (no pre-existing vector files required).
PATH_TT_USER_VECS = TWO_TOWER_DIR / "user_vectors.npz"
PATH_TT_ITEM_VECS = TWO_TOWER_DIR / "item_vectors.npz"

PATH_COMPARE_OVERALL_OUT = BASELINE_EVAL_DIR / "eval_retrieval_with_two_tower_overall.csv"
PATH_COMPARE_BY_SLICE_OUT = BASELINE_EVAL_DIR / "eval_retrieval_with_two_tower_by_slice.csv"

print("Optional vector export dir:", TWO_TOWER_DIR)
print("Optional vector outputs:")
print(" -", PATH_TT_USER_VECS)
print(" -", PATH_TT_ITEM_VECS)
print("Comparison table outputs:")
print(" -", PATH_COMPARE_OVERALL_OUT)
print(" -", PATH_COMPARE_BY_SLICE_OUT)

Optional vector export dir: /home/ryanr/workspace/steam_recommendations/artifacts/recs/two_tower
Optional vector outputs:
 - /home/ryanr/workspace/steam_recommendations/artifacts/recs/two_tower/user_vectors.npz
 - /home/ryanr/workspace/steam_recommendations/artifacts/recs/two_tower/item_vectors.npz
Comparison table outputs:
 - /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval/eval_retrieval_with_two_tower_overall.csv
 - /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval/eval_retrieval_with_two_tower_by_slice.csv


### Local scoring + metric helpers

This cell adds notebook-local helpers that mirror the centralized metric contract (`Hit@K`, `Recall@K`, `MAP@K`, `NDCG@K`, `MRR`) without depending on private functions.

In [17]:
from typing import Any

import numpy as np

from steam_review_ml.recommender.evaluation import (
    average_precision_at_k,
    hit_rate_at_k,
    mrr,
    ndcg_at_k,
    recall_at_k,
)


def _rank_rows(scores: np.ndarray) -> np.ndarray:
    if scores.ndim != 1:
        raise ValueError(f"Expected 1D score vector, got shape={scores.shape}")
    if np.isnan(scores).any():
        scores = np.nan_to_num(scores, nan=-np.inf)
    return np.argsort(-scores, kind="stable")


def _slice_name_from_n_targets(n_eval_targets: int) -> str:
    if n_eval_targets >= 2:
        return "slice_a_multi_target"
    if n_eval_targets == 1:
        return "slice_b_single_target"
    if n_eval_targets == 0:
        return "slice_c_zero_target"
    return "slice_other"


def _compute_metric_row(
    *,
    method: str,
    ranked_rows: np.ndarray,
    positives: set[int],
    app_ids: np.ndarray,
    k_final: int,
    n_eval_targets: int,
) -> dict[str, Any]:
    return {
        "method": method,
        "n_eval_targets": int(n_eval_targets),
        "slice_name": _slice_name_from_n_targets(int(n_eval_targets)),
        "Hit@K": hit_rate_at_k(ranked_rows, positives, k_final, app_ids),
        "Recall@K": recall_at_k(ranked_rows, positives, k_final, app_ids),
        "MAP@K": average_precision_at_k(ranked_rows, positives, k_final, app_ids),
        "NDCG@K": ndcg_at_k(ranked_rows, positives, k_final, app_ids),
        "MRR": mrr(ranked_rows, positives, app_ids),
    }

## Temporary check: normalized playtime fields in processed train split

This is a temporary preflight check to confirm `_norm_*` playtime features exist before loading eval examples.

In [18]:
CHECK_FOR_NORM_COLS = False

if CHECK_FOR_NORM_COLS:
    TRAIN_NORM_PATH = REPO_ROOT / "data" / "processed" / "steam_reviews_cleaned_english_train_norm.parquet"
    VAL_NORM_PATH = REPO_ROOT / "data" / "processed" / "steam_reviews_cleaned_english_val_norm.parquet"
    CHECK_NORM_COLS = [
        "_norm_author__playtime_at_review",
        "_norm_author__playtime_last_two_weeks",
        "_norm_author__num_games_owned",
        "_norm_author__num_reviews",
        "_norm_review_word_count",
    ]

    if not TRAIN_NORM_PATH.is_file():
        raise FileNotFoundError(f"Missing processed train parquet: {TRAIN_NORM_PATH}")
    if not VAL_NORM_PATH.is_file():
        raise FileNotFoundError(f"Missing processed val parquet: {VAL_NORM_PATH}")

    df_tmp = pd.read_parquet(TRAIN_NORM_PATH)
    df_tmp_val = pd.read_parquet(VAL_NORM_PATH)
    def check_norm_cols(df, df_name, cols, file_path):
        present = [c for c in cols if c in df.columns]
        missing = [c for c in cols if c not in df.columns]
        print(f"Processed {df_name} file:", file_path)
        print("Present normalized columns:", present)
        print("Missing normalized columns:", missing)
        if present:
            non_null_counts = {c: int(df[c].notna().sum()) for c in present}
            print("Non-null counts:", non_null_counts)
            print(df[present].head(3))

    check_norm_cols(df_tmp, "train", CHECK_NORM_COLS, TRAIN_NORM_PATH)
    check_norm_cols(df_tmp_val, "val", CHECK_NORM_COLS, VAL_NORM_PATH)
else:
    print("Skipping normalized column check.")


Skipping normalized column check.


## Load eval cohort and two-tower artifacts

This cell builds the same eval examples used by the baseline job (`prepare_eval_inputs`) and loads two-tower vectors + id maps. It fails early on shape/id mismatches to avoid silent metric drift.

In [19]:
from collections import Counter, defaultdict

from steam_review_ml.constants import PROJECT_RANDOM_SEED
from steam_review_ml.recommender.evaluation import prepare_eval_inputs
from steam_review_ml.recommender.math_utils import l2_normalize
from steam_review_ml.recommender.retrieve import ContentRetriever

# Clean conceptual split:
# - Behavior weighting uses only game-tied playtime-at-review.
# - Context scalars are user/activity features kept separate.
PLAYTIME_CANDIDATE_KEYS = ("_norm_author__playtime_at_review",)
CONTEXT_SCALAR_KEYS = (
    "_norm_author__playtime_last_two_weeks",
    "_norm_author__num_games_owned",
    "_norm_author__num_reviews",
)

def _validate_eval_config(cfg: dict) -> None:
    """
    Ensure all required keys are present in the evaluation config.
    This function guards against missing or malformed configs at runtime.
    """
    required_keys = (
        "split",
        "active_cohort",
        "max_examples",
        "support_app_filter_mode",
        "min_review_chars",
        "max_train_rows_per_user",
    )
    missing = [k for k in required_keys if k not in cfg]
    if missing:
        raise ValueError(f"Missing required eval config keys: {missing}")

def _playtime_key_coverage(rows: list[dict]) -> dict[str, int]:
    """
    Count how often each candidate playtime key appears with non-null values.
    """
    counts = Counter()
    for row in rows:
        for key in PLAYTIME_CANDIDATE_KEYS:
            if key in row and row[key] is not None:
                counts[key] += 1
    return {k: int(counts[k]) for k in PLAYTIME_CANDIDATE_KEYS}


def _coverage(rows: list[dict], keys: tuple[str, ...]) -> dict[str, int]:
    """Generic non-null key coverage counter for row dicts."""
    counts = Counter()
    for row in rows:
        for key in keys:
            if key in row and row[key] is not None:
                counts[key] += 1
    return {k: int(counts[k]) for k in keys}


def _validate_playtime_fields(rows: list[dict]) -> dict[str, int]:
    """
    Diagnostic-only validation for playtime fields.
    Prints coverage and available keys; does not raise.
    """
    if not rows:
        print("Warning: no train review rows found; behavior weighting will use fallback.")
        return {k: 0 for k in PLAYTIME_CANDIDATE_KEYS}

    coverage = _playtime_key_coverage(rows)
    sample_keys = sorted(rows[0].keys()) if rows else []

    if not any(coverage.values()):
        print(
            "Warning: no usable playtime fields found in train_review_rows. "
            f"Checked keys: {list(PLAYTIME_CANDIDATE_KEYS)}"
        )
        print("Available keys in train_review_rows (sample):", sample_keys)
        print("Using unweighted behavior fallback where needed.")
    else:
        print("Playtime key coverage:", coverage)

    return coverage

def _extract_playtime_weight(row: dict) -> float:
    """
    Extract pre-normalized playtime weight from a review row.
    Uses the maximum positive value across candidate normalized fields.
    """
    vals = []
    for key in PLAYTIME_CANDIDATE_KEYS:
        if key in row and row[key] is not None:
            vals.append(max(0.0, float(row[key])))
    if not vals:
        return 0.0
    return float(max(vals))

def _build_u_reviews(ex: dict, *, retriever) -> np.ndarray:
    """Text-history embedding, independent of behavior weighting."""
    support_texts = [
        str(r.get("text", "")).strip()
        for r in ex.get("train_review_rows", [])
        if str(r.get("text", "")).strip()
    ]
    if support_texts:
        vecs = np.stack([retriever.embed_text(t) for t in support_texts], axis=0).astype(np.float32)
        return l2_normalize(vecs.mean(axis=0))
    return retriever.embed_text(str(ex.get("query_text", "")))

def _build_u_behavior_weighted(
    ex: dict,
    *,
    X: np.ndarray,
    app_to_row: dict[int, int],
    fallback: np.ndarray,
) -> tuple[np.ndarray, bool]:
    """Per-game behavior embedding weighted by playtime-at-review only."""
    rows = ex.get("train_review_rows", [])
    weighted_vecs = []
    weights = []
    for r in rows:
        app_id = int(r.get("app_id", -1))
        row_idx = app_to_row.get(app_id)
        if row_idx is None:
            continue  # No embedding available for this app.
        w = _extract_playtime_weight(r)
        if w <= 0.0:
            continue
        weighted_vecs.append(X[row_idx].astype(np.float32))
        weights.append(w)

    if weighted_vecs:
        mat = np.stack(weighted_vecs, axis=0)
        w_arr = np.asarray(weights, dtype=np.float32)
        vec = (mat * w_arr[:, None]).sum(axis=0) / np.maximum(w_arr.sum(), 1e-12)
        return l2_normalize(vec), True

    # Fallback retained for robustness; explicitly tracked in diagnostics.
    support_app_ids = sorted({int(r.get("app_id")) for r in rows if int(r.get("app_id", -1)) in app_to_row})
    if support_app_ids:
        mat = np.stack([X[app_to_row[a]] for a in support_app_ids], axis=0).astype(np.float32)
        return l2_normalize(mat.mean(axis=0)), False

    return fallback, False

def _build_user_context_scalars(ex: dict) -> dict[str, float]:
    """Current/user context scalars are kept separate from behavior weighting."""
    rows = ex.get("train_review_rows", [])
    out: dict[str, float] = {}
    for key in CONTEXT_SCALAR_KEYS:
        vals = [float(r[key]) for r in rows if key in r and r[key] is not None]
        out[key] = float(np.mean(vals)) if vals else 0.0
    return out


def _load_examples_from_cache(path: Path) -> list[dict]:
    """Load eval examples cache written by recs_job_build_eval_examples.py."""
    df = pd.read_parquet(path)
    required_cols = {
        "user_id",
        "query_app_id",
        "query_text",
        "query_ts",
        "n_eval_targets",
        "cohort",
        "eval_pos_cohort",
        "positives_json",
        "train_review_rows_json",
    }
    missing = sorted(c for c in required_cols if c not in df.columns)
    if missing:
        raise ValueError(f"Cached examples missing required columns: {missing}")

    examples: list[dict] = []
    for row in df.itertuples(index=False):
        rec = row._asdict()
        examples.append(
            {
                "user_id": str(rec["user_id"]),
                "query_app_id": int(rec["query_app_id"]),
                "query_text": str(rec["query_text"]),
                "query_ts": float(rec["query_ts"]),
                "positives": set(int(a) for a in json.loads(rec["positives_json"])),
                "n_eval_targets": int(rec["n_eval_targets"]),
                "train_review_rows": json.loads(rec["train_review_rows_json"]),
                "cohort": str(rec["cohort"]),
                "eval_pos_cohort": str(rec["eval_pos_cohort"]),
            }
        )
    return examples


In [20]:
# 1) Validate config and choose examples source (cache-first, fallback to prepare_eval_inputs)
print("Loading config and validating required keys...")
cfg = load_config(str(EVAL_JOB_CONFIG))
_validate_eval_config(cfg)
print("Config loaded and validated.")

USE_EXAMPLES_CACHE = True
EVAL_EXAMPLES_CACHE = (
    REPO_ROOT
    / "artifacts"
    / "recs"
    / "eval_cache"
    / "val_dev_12k_v1"
    / "eval_examples.parquet"
)

print("Loading shared retriever/item embeddings...")
retriever = ContentRetriever(repo_root=REPO_ROOT)
catalog_app_ids = np.asarray(retriever.app_ids)
X = np.asarray(retriever.embedding_matrix)
app_to_row = {int(a): i for i, a in enumerate(catalog_app_ids.tolist())}

if X.ndim != 2:
    raise ValueError(f"Expected 2D embedding matrix, got shape={X.shape}")
if len(catalog_app_ids) != X.shape[0]:
    raise ValueError("app_ids length must match embedding matrix rows")

cache_loaded = False
if USE_EXAMPLES_CACHE and EVAL_EXAMPLES_CACHE.is_file():
    print(f"Loading eval examples from cache: {EVAL_EXAMPLES_CACHE}")
    examples = _load_examples_from_cache(EVAL_EXAMPLES_CACHE)
    cache_loaded = True
else:
    if USE_EXAMPLES_CACHE:
        print(f"Cache not found at {EVAL_EXAMPLES_CACHE}; falling back to prepare_eval_inputs().")
    else:
        print("USE_EXAMPLES_CACHE=False; using prepare_eval_inputs().")

    prepared = prepare_eval_inputs(
        repo_root=REPO_ROOT,
        split=str(cfg.get("split", "val")),
        active_cohort=str(cfg.get("active_cohort", "all")),
        max_examples=int(cfg.get("max_examples", 12_500)),
        support_app_filter_mode=str(cfg.get("support_app_filter_mode", "strict")),
        cohort_sizing={
            tuple(k.split("|", 1)): float(v)
            for k, v in dict(cfg.get("cohort_sizing", {})).items()
        },
        min_review_chars=int(cfg.get("min_review_chars", 30)),
        max_train_rows_per_user=int(cfg.get("max_train_rows_per_user", 5)),
        random_seed=int(cfg.get("random_seed", PROJECT_RANDOM_SEED)),
        artifact_dir=REPO_ROOT / str(cfg.get("artifact_dir", "artifacts/recs")),
        verbose=False,
    )
    examples = prepared.examples

if not examples:
    raise RuntimeError("No evaluation examples available from cache or prepare_eval_inputs")

from types import SimpleNamespace

inputs = SimpleNamespace(
    examples=examples,
    retriever=retriever,
    app_ids=catalog_app_ids,
    embedding_matrix=X,
    app_to_row=app_to_row,
    source="cache" if cache_loaded else "prepare_eval_inputs",
)

print(f"Loaded {len(inputs.examples)} eval examples from {inputs.source}.")
print(f"Catalog embeddings: {X.shape[0]} items x {X.shape[1]} dims.")

# 2) Prepare catalog/item matrix
catalog_item_matrix = X.astype(np.float32, copy=False)
print("Catalog item matrix ready. Shape:", catalog_item_matrix.shape)

# 3) Validate behavior/context field availability
print("Validating behavior/context fields on train review rows...")
all_train_rows = [r for ex in inputs.examples for r in ex.get("train_review_rows", [])]
_ = _validate_playtime_fields(all_train_rows)
context_coverage = _coverage(all_train_rows, CONTEXT_SCALAR_KEYS)
print("Context scalar key coverage:", context_coverage)
positive_weight_rows = sum(1 for r in all_train_rows if _extract_playtime_weight(r) > 0.0)
print(f"Rows with positive behavior weight ({PLAYTIME_CANDIDATE_KEYS[0]}): {positive_weight_rows}/{len(all_train_rows)}")

# 4) Build user vectors
user_vec_lists: dict[str, list[np.ndarray]] = defaultdict(list)
vector_rows = []
n_weighted_examples = 0
n_fallback_examples = 0

for ex_idx, ex in enumerate(inputs.examples):
    u_reviews = _build_u_reviews(ex, retriever=inputs.retriever)
    u_behavior, used_weighted = _build_u_behavior_weighted(ex, X=X, app_to_row=app_to_row, fallback=u_reviews)
    if used_weighted:
        n_weighted_examples += 1
    else:
        n_fallback_examples += 1

    u_habit = l2_normalize(0.5 * u_behavior + 0.5 * u_reviews)
    context_scalars = _build_user_context_scalars(ex)

    user_id = str(ex.get("user_id"))
    user_vec_lists[user_id].append(u_habit.astype(np.float32))

    vector_rows.append(
        {
            "ex_idx": ex_idx,
            "user_id": user_id,
            "query_app_id": int(ex.get("query_app_id")),
            "u_reviews": u_reviews,
            "u_behavior": u_behavior,
            "u_habit": u_habit,
            **context_scalars,
        }
    )

user_vec_by_id = {
    uid: l2_normalize(np.stack(vecs, axis=0).mean(axis=0)).astype(np.float32)
    for uid, vecs in user_vec_lists.items()
}

print("Generated in-notebook user vectors")
print("  examples:", len(inputs.examples))
print("  users with vectors:", len(user_vec_by_id))
print("  weighted behavior examples:", n_weighted_examples)
print("  fallback behavior examples:", n_fallback_examples)
print("  catalog size:", len(catalog_app_ids))
print("  embedding dimension:", catalog_item_matrix.shape[1])

Loading config and validating required keys...
Config loaded and validated.
Loading shared retriever/item embeddings...
Loading eval examples from cache: /home/ryanr/workspace/steam_recommendations/artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet
Loaded 12500 eval examples from cache.
Catalog embeddings: 315 items x 512 dims.
Catalog item matrix ready. Shape: (315, 512)
Validating behavior/context fields on train review rows...
Playtime key coverage: {'_norm_author__playtime_at_review': 24032}
Context scalar key coverage: {'_norm_author__playtime_last_two_weeks': 24032, '_norm_author__num_games_owned': 24032, '_norm_author__num_reviews': 24032}
Rows with positive behavior weight (_norm_author__playtime_at_review): 23984/24032
Generated in-notebook user vectors
  examples: 12500
  users with vectors: 12265
  weighted behavior examples: 9370
  fallback behavior examples: 3130
  catalog size: 315
  embedding dimension: 512


In [21]:
pd.DataFrame(all_train_rows).head()

,app_id,text,ts,_norm_author__playtime_at_review,_norm_author__playtime_last_two_weeks,_norm_author__num_games_owned,_norm_author__num_reviews,_norm_review_word_count
0,359550,Facepalm as your teammates run into marked ene...,1.473255e+09,7.486613,0.0,6.906755,2.995732,3.044522
1,571740,> Download Game\n> Get some rage maps off the ...,1.542887e+09,6.285998,0.0,6.907755,2.995732,2.995732
2,899440,"As a long term Monster Hunter fan, this itches...",1.549898e+09,7.702556,0.0,5.579730,2.484907,3.988984
3,945360,"Claimed someone was a murder, followed her aro...",1.600532e+09,4.369448,0.0,5.579730,2.484907,3.465736
4,582010,As a Monster Hunter fan from the PSP / PS2 era...,1.534964e+09,8.237479,0.0,5.579730,2.484907,4.043051


## Cell 13 — Score examples and aggregate contract tables

This cell computes per-example scores via user-vector dot product against catalog item vectors, then produces:
- `two_tower_overall`
- `two_tower_by_slice`

with the same core metric columns as baseline outputs.

In [25]:
inputs.examples[0]

{'user_id': '76561198001296435',
 'query_app_id': 812140,
 'query_text': "> Got it on Sale\n> Started it up, played Kassandra 'coz better voice acting imo\n> Played a bit, unlock boat\n> Destroyed a bunch of ships\n> Rammed a bunch of ships\n> Boarded a bunch of ships\n> Wait there's more story after you unlock the boat?\n> Meh, RAMMING SPEED!!!",
 'query_ts': 1568427802.0,
 'positives': {262060},
 'n_eval_targets': 1,
 'train_review_rows': [{'app_id': 359550,
   'text': 'Facepalm as your teammates run into marked enemies before being outnumbered and killed yourself, followed by a complementary teabagging. 11/10',
   'ts': 1473255297.0,
   '_norm_author__playtime_at_review': 7.486613313139955,
   '_norm_author__playtime_last_two_weeks': 0.0,
   '_norm_author__num_games_owned': 6.906754778648554,
   '_norm_author__num_reviews': 2.995732273553991,
   '_norm_review_word_count': 3.044522437723423},
  {'app_id': 571740,
   'text': '> Download Game\n> Get some rage maps off the workshop\n> R

In [22]:
rows = []
skipped_no_user_vector = 0

for ex in inputs.examples:
    positives = set(int(a) for a in ex.get("positives", set()))
    if not positives:
        continue

    user_id = str(ex.get("user_id"))
    q_app = int(ex.get("query_app_id"))
    n_eval_targets = int(ex.get("n_eval_targets", len(positives)))

    u = user_vec_by_id.get(user_id)
    if u is None:
        skipped_no_user_vector += 1
        continue

    scores = catalog_item_matrix @ u.astype(np.float32)

    # Keep query app masked to match baseline retrieval evaluation behavior.
    q_idx = inputs.app_to_row.get(q_app)
    if q_idx is not None:
        scores[q_idx] = -np.inf

    ranked_rows = _rank_rows(scores)
    metric_row = _compute_metric_row(
        method=CANDIDATE_METHOD,
        ranked_rows=ranked_rows,
        positives=positives,
        app_ids=catalog_app_ids,
        k_final=K_FINAL,
        n_eval_targets=n_eval_targets,
    )
    rows.append(metric_row)

if not rows:
    raise RuntimeError("No two-tower rows scored. Check user id alignment and vector artifacts.")

df_tt_ex = pd.DataFrame(rows)
two_tower_overall = (
    df_tt_ex.groupby("method", observed=True)[METRIC_COLS]
    .mean()
    .reset_index()
)
two_tower_by_slice = (
    df_tt_ex.groupby(["slice_name", "method"], observed=True)[METRIC_COLS]
    .mean()
    .reset_index()
    .sort_values(["slice_name", "NDCG@K", "Hit@K"], ascending=[True, False, False])
    .reset_index(drop=True)
)

print("Two-tower scoring complete")
print(" scored rows:", len(df_tt_ex))
print(" skipped (missing user vector):", skipped_no_user_vector)
display(two_tower_overall)
display(two_tower_by_slice)

Two-tower scoring complete
 scored rows: 12500
 skipped (missing user vector): 0


,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR
0,two_tower,0.08872,0.082657,0.025796,0.039525,0.042815


,slice_name,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR
0,slice_a_multi_target,two_tower,0.173793,0.069250,0.022710,0.044229,0.079190
1,slice_b_single_target,two_tower,0.083482,0.083482,0.025986,0.039235,0.040576


## Cell 15 — Merge with baseline and optionally write outputs

This cell concatenates baseline and candidate tables and can save them as new files without mutating the original baseline artifacts.

In [23]:
WRITE_OUTPUTS = False

baseline_by_slice_sub = baseline_by_slice[baseline_by_slice["method"].isin(BASELINE_METHODS)].copy()

compare_overall_full = pd.concat(
    [baseline_overall_sub[required], two_tower_overall[required]],
    ignore_index=True,
)
compare_by_slice_full = pd.concat(
    [
        baseline_by_slice_sub[["slice_name", "method", *METRIC_COLS]],
        two_tower_by_slice[["slice_name", "method", *METRIC_COLS]],
    ],
    ignore_index=True,
)

print("Combined overall table:")
display(compare_overall_full.sort_values("NDCG@K", ascending=False).reset_index(drop=True))

print("\nCombined by-slice table:")
display(compare_by_slice_full.sort_values(["slice_name", "NDCG@K"], ascending=[True, False]).reset_index(drop=True))

if WRITE_OUTPUTS:
    compare_overall_full.to_csv(PATH_COMPARE_OVERALL_OUT, index=False)
    compare_by_slice_full.to_csv(PATH_COMPARE_BY_SLICE_OUT, index=False)
    print("\nWrote:")
    print(" -", PATH_COMPARE_OVERALL_OUT)
    print(" -", PATH_COMPARE_BY_SLICE_OUT)
else:
    print("\nWRITE_OUTPUTS=False, no files written.")

Combined overall table:


,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR
0,popularity_train,0.15112,0.146796,0.050594,0.073109,0.073756
1,two_tower,0.08872,0.082657,0.025796,0.039525,0.042815
2,multi_mean_train,0.08328,0.076479,0.025349,0.037700,0.041694
3,raw,0.07168,0.066606,0.024917,0.035020,0.040381



Combined by-slice table:


,slice_name,method,Hit@K,Recall@K,MAP@K,NDCG@K,MRR
0,slice_a_multi_target,two_tower,0.173793,0.069250,0.022710,0.044229,0.079190
1,slice_a_multi_target,multi_mean_train,0.188966,0.071700,0.021733,0.044159,0.075772
2,slice_a_multi_target,raw,0.143448,0.055964,0.021677,0.038812,0.073125
3,slice_a_multi_target,popularity_train,0.128276,0.053728,0.019418,0.035444,0.076750
4,slice_b_single_target,popularity_train,0.152527,0.152527,0.052514,0.075428,0.073572
5,slice_b_single_target,two_tower,0.083482,0.083482,0.025986,0.039235,0.040576
6,slice_b_single_target,multi_mean_train,0.076773,0.076773,0.025572,0.037302,0.039596
7,slice_b_single_target,raw,0.067261,0.067261,0.025117,0.034786,0.038365



WRITE_OUTPUTS=False, no files written.


## Step 1 — Build retrieval vectors first (`u_reviews`, `u_behavior`)

This block runs without two-tower `.npz` artifacts. It builds evaluation examples, then constructs:
- `u_reviews`: pooled train-support review-text embedding
- `u_behavior`: playtime-weighted pooled train-support app embedding (`w_i = log1p(hours_i)`)

These vectors are the building blocks for candidates C/D.

In [24]:
# Deprecated duplicate cell kept as no-op for notebook stability.
pass